In [1]:
import pandas as pd
import numpy as np

In [2]:
naps_csv = r"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\all_NAPs_ratings.csv"

In [5]:
df_naps = pd.read_csv(naps_csv)
df = df_naps.copy()

In [3]:
N_BINS = 8
TRIALS_PER_BIN = 14
COND_PER_BIN = 7
SEED = 42
np.random.seed(SEED)

In [6]:
bins = [1, 2, 3, 4, 5, 6, 7, 8, 9]
bin_labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins) - 1)]

df["val_bin"] = pd.cut(
    df["Valence"],
    bins=bins,
    labels=bin_labels,
    include_lowest=True
)

counts = df["val_bin"].value_counts().reindex(bin_labels)
print(counts)

val_bin
1-2     22
2-3    113
3-4    194
4-5    200
5-6    248
6-7    334
7-8    229
8-9     16
Name: count, dtype: int64


In [7]:
# ===== DEFINE PER-BIN TARGETS (FIXED) =====

sessions = 5
TARGET_TOTAL = 112

bin_counts = df["val_bin"].value_counts().reindex(bin_labels).to_dict()

per_session_targets = {}

# ---- Step 1: fixed sparse bins ----
per_session_targets["1-2"] = bin_counts["1-2"] // sessions   # ~5
per_session_targets["8-9"] = bin_counts["8-9"] // sessions   # ~12

# ---- Step 2: remaining slots ----
remaining = TARGET_TOTAL - (per_session_targets["1-2"] + per_session_targets["8-9"])

middle_bins = [b for b in bin_labels if b not in ["1-2", "8-9"]]

base = remaining // len(middle_bins)   # ~15
extra = remaining % len(middle_bins)   # distribute +1

# ---- Step 3: assign evenly ----
for i, b in enumerate(middle_bins):
    per_session_targets[b] = base + (1 if i < extra else 0)

print("Per-session targets:", per_session_targets)
print("Total per session:", sum(per_session_targets.values()))

Per-session targets: {'1-2': 5, '8-9': 12, '2-3': 16, '3-4': 16, '4-5': 16, '5-6': 16, '6-7': 16, '7-8': 15}
Total per session: 112


In [8]:
# ===== PREPARE BIN POOLS =====


bin_pools = {}

for b in bin_labels:
    pool = df[df["val_bin"] == b].copy()
    pool = pool.sample(frac=1, random_state=SEED).reset_index(drop=True)
    bin_pools[b] = pool

In [9]:
# ===== BUILD SESSIONS =====

sessions_data = []

for s in range(sessions):
    session_samples = []

    for b in bin_labels:
        n_take = per_session_targets[b]

        pool = bin_pools[b]

        # take top n samples
        take = pool.iloc[:n_take]
        session_samples.append(take)

        # remove used samples (no reuse)
        bin_pools[b] = pool.iloc[n_take:].reset_index(drop=True)

    session_df = pd.concat(session_samples).reset_index(drop=True)

    # shuffle session
    session_df = session_df.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

    sessions_data.append(session_df)

In [10]:
assert len(session_df) == 112
print(session_df["val_bin"].value_counts().sort_index())

val_bin
1-2     5
2-3    16
3-4    16
4-5    16
5-6    16
6-7    16
7-8    15
8-9    12
Name: count, dtype: int64


In [11]:
# ===== ASSIGN CONDITIONS WITHIN EACH BIN =====

for s, session_df in enumerate(sessions_data):

    session_df = session_df.copy()
    parts = []

    for b in bin_labels:
        bin_df = session_df[session_df["val_bin"] == b].copy()

        n = len(bin_df)
        half = n // 2

        # assign conditions within bin
        conds = ["FEEL"] * half + ["TONE"] * (n - half)

        # shuffle within bin before assigning
        bin_df = bin_df.sample(frac=1, random_state=SEED + s).reset_index(drop=True)
        bin_df["condition"] = conds

        parts.append(bin_df)

    # combine all bins
    session_df = pd.concat(parts).reset_index(drop=True)

    # final shuffle across whole session
    session_df = session_df.sample(frac=1, random_state=SEED + s).reset_index(drop=True)

    sessions_data[s] = session_df

In [12]:
# ===== SAVE SESSIONS =====

for i, session_df in enumerate(sessions_data):
    out_path = rf"C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\session_{i+2}_block_112.csv"
    session_df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")

Saved: C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\session_2_block_112.csv
Saved: C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\session_3_block_112.csv
Saved: C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\session_4_block_112.csv
Saved: C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\session_5_block_112.csv
Saved: C:\Users\fkamdar\Desktop\repos\pd_nonmotor\stimuli\blocks\session_6_block_112.csv


In [17]:
import os
import shutil

src_dir = r"C:\Users\fkamdar\Desktop\IAPS\IAPS 1-20 Images"
dst_dir = r"C:\Users\fkamdar\Desktop\IAPS\Extreme_Images"

os.makedirs(dst_dir, exist_ok=True)

# assuming df_iaps already filtered for extreme bins
image_ids = df_iaps["ID"].astype(int).astype(str).tolist()
copied = 0
missing = []

for img_id in image_ids:
    found = False
    
    # try common extensions
    for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
        src_path = os.path.join(src_dir, img_id + ext)
        
        if os.path.exists(src_path):
            shutil.copy(src_path, os.path.join(dst_dir, img_id + ext))
            copied += 1
            found = True
            break
    
    if not found:
        missing.append(img_id)

print(f"Copied: {copied}")
print(f"Missing: {len(missing)}")

Copied: 80
Missing: 0
